# Notebook Header & Imports

In [20]:
# EEEM068: Industrial Waste Classification
# Training Notebook (Jupyter version of train.py)

import os
import json
import copy
import argparse
import random
import time
from pathlib import Path

import yaml
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Markdown: Config Loading

In [21]:
## 🔧 Configuration Loading

#This section loads:
#- `base.yaml`
#- experiment YAML
#- merges them (experiment overrides base)
#- applies optional overrides (learning rate, batch size, epochs)

#This mirrors the behaviour of `train.py`.

# Config Functions

In [22]:
def deep_merge(base: dict, override: dict) -> dict:
    result = copy.deepcopy(base)
    for key, value in override.items():
        if key == "base":
            continue
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = value
    return result


def load_config(experiment_path: str, cli_overrides: dict = None) -> dict:
    with open(experiment_path) as f:
        experiment = yaml.safe_load(f)

    if "base" in experiment:
        with open(experiment["base"]) as f:
            base = yaml.safe_load(f)
        config = deep_merge(base, experiment)
    else:
        config = experiment

    if cli_overrides:
        for key, value in cli_overrides.items():
            if value is not None:
                config["training"][key] = value

    return config


def save_config(config: dict, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "config.yaml"), "w") as f:
        yaml.dump(config, f)

# Markdown: Reproducibility

In [23]:
## 🎲 Reproducibility

#Fix all random seeds for:
#- Python
#- NumPy
#- PyTorch CPU
#- PyTorch CUDA

#This ensures deterministic behaviour.

# Seed Function

In [24]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[Seed] Set to {seed}")

# Markdown: Data Transforms & Loaders

In [25]:
## Data Transforms & DataLoaders

#This section builds:
#- training transforms (with augmentation)
#- validation/test transforms
#- ImageFolder datasets
#- DataLoaders with correct workers + pin_memory
#- class weights for imbalance

# Transforms + DataLoaders

In [26]:
def get_transforms(config: dict, train: bool = True):
    aug = config["augmentation"]
    img_size = (config["dataset"]["img_height"], config["dataset"]["img_width"])

    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size),
            transforms.RandomHorizontalFlip() if aug["random_horizontal_flip"] else transforms.Lambda(lambda x: x),
            transforms.ColorJitter(
                brightness=aug["brightness"],
                contrast=aug["contrast"],
                saturation=aug["saturation"],
                hue=aug["hue"]
            ),
            transforms.ToTensor(),
            normalize
        ])
    else:
        return transforms.Compose([
            transforms.Resize(img_size),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            normalize
        ])


def get_dataloaders(config: dict):
    train_tf = get_transforms(config, train=True)
    val_tf   = get_transforms(config, train=False)

    data_root  = config["dataset"]["data_root"]
    batch_size = config["training"]["batch_size"]
    seed       = config["dataset"]["seed"]

    train_dataset = datasets.ImageFolder(os.path.join(data_root, "train"), transform=train_tf)
    val_dataset   = datasets.ImageFolder(os.path.join(data_root, "val"),   transform=val_tf)
    test_dataset  = datasets.ImageFolder(os.path.join(data_root, "test"),  transform=val_tf)

    class_counts  = np.array([train_dataset.targets.count(i)
                               for i in range(config["dataset"]["n_classes"])])
    class_weights = torch.FloatTensor(1.0 / (class_counts + 1e-6))

    pin = torch.cuda.is_available()
    workers = 0 if os.name == "nt" else 4

    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=workers, pin_memory=pin, generator=g)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=workers, pin_memory=pin)
    test_loader = DataLoader(test_dataset, batch_size=config["evaluation"]["batch_size"],
                             shuffle=False, num_workers=workers, pin_memory=pin)

    print(f"[Data] Train={len(train_dataset)} | Val={len(val_dataset)} | Test={len(test_dataset)}")
    return train_loader, val_loader, test_loader, class_weights

# Markdown: Model Loading

In [27]:
## 🧠 Model Loader

#Supports:
#- ResNet‑50  
#- EfficientNet‑B3  
#- Swin‑T  
#- ConvNeXt‑Tiny  

#Replaces classification head with correct number of classes.

# Model Builder

In [28]:
def get_model(config: dict):
    backbone  = config["run"]["model"]
    n_classes = config["dataset"]["n_classes"]
    pretrained = config["training"]["pretrained"]
    weights = "IMAGENET1K_V1" if pretrained else None

    print(f"[Model] Loading {backbone} (pretrained={pretrained})")

    if backbone == "resnet50":
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, n_classes)

    elif backbone == "efficientnet_b3":
        model = models.efficientnet_b3(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, n_classes)

    elif backbone == "swin_t":
        model = models.swin_t(weights=weights)
        model.head = nn.Linear(model.head.in_features, n_classes)

    elif backbone == "convnext_t":
        model = models.convnext_tiny(weights=weights)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, n_classes)

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[Model] Params — total: {total:,} | trainable: {trainable:,}")

    return model

# Markdown: Optimizer + Scheduler

In [29]:
## ⚙️ Optimizer & Scheduler

#Supports:
#- Adam
#- SGD
#- staged learning rates (backbone vs head)
#- CosineAnnealingLR
#- MultiStepLR

# Optimizer + Scheduler

In [30]:
def get_optimizer(model, config):
    opt_cfg = config["optimizer"]
    ft_cfg  = config["fine_tuning"]
    lr      = config["training"]["learning_rate"]

    if ft_cfg["staged_lr"]:
        new_layer_names = ft_cfg.get("new_layers", ["fc", "classifier", "head"])
        head_params, backbone_params = [], []

        for name, param in model.named_parameters():
            if any(layer in name for layer in new_layer_names):
                head_params.append(param)
            else:
                backbone_params.append(param)

        param_groups = [
            {"params": backbone_params, "lr": lr * ft_cfg["base_lr_mult"]},
            {"params": head_params,     "lr": lr}
        ]
    else:
        param_groups = model.parameters()

    if opt_cfg["type"] == "adam":
        return torch.optim.Adam(param_groups, lr=lr,
                                betas=(opt_cfg["adam_beta1"], opt_cfg["adam_beta2"]),
                                weight_decay=opt_cfg["weight_decay"])
    else:
        return torch.optim.SGD(param_groups, lr=lr,
                               momentum=opt_cfg["momentum"],
                               dampening=opt_cfg["sgd_dampening"],
                               nesterov=opt_cfg["sgd_nesterov"],
                               weight_decay=opt_cfg["weight_decay"])


def get_scheduler(optimizer, config):
    sched_type = config["scheduler"]["type"]
    ft_cfg = config["fine_tuning"]

    if sched_type == "CosineAnnealingLR":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ft_cfg["T_max"])
    else:
        return torch.optim.lr_scheduler.MultiStepLR(
            optimizer, milestones=ft_cfg["stepsize"], gamma=ft_cfg["gamma"]
        )

# Markdown: Training & Validation

In [31]:
## 🏋️ Training & Validation Loops

#These functions:
#- train for one epoch
#- validate on val set
#- compute accuracy + F1

# Training + Validation

In [32]:
def train_one_epoch(model, loader, criterion, optimizer, device, log_interval):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += images.size(0)

        if (batch_idx + 1) % log_interval == 0:
            print(f"[Batch {batch_idx+1}] loss={loss.item():.4f}")

    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / total, correct / total, f1, all_preds, all_labels

# Markdown: Plotting & Saving


In [33]:
## 📊 Saving Metrics & Plots

#This includes:
#- metrics.json
#- loss curves
#- confusion matrix

# Plotting Functions

In [34]:
def save_metrics(metrics, output_dir):
    with open(os.path.join(output_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)


def save_loss_curves(metrics, output_dir):
    epochs = range(1, len(metrics["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, metrics["train_loss"], label="Train")
    ax1.plot(epochs, metrics["val_loss"], label="Val")
    ax1.set_title("Loss")
    ax1.legend()

    ax2.plot(epochs, metrics["train_acc"], label="Train")
    ax2.plot(epochs, metrics["val_acc"], label="Val")
    ax2.set_title("Accuracy")
    ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "loss_curves.png"))
    plt.close()


def save_confusion_matrix(labels, preds, class_names, output_dir):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(20, 18))
    sns.heatmap(cm, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "confusion_matrix.png"))
    plt.close()

# Markdown: Main Training Cel

In [35]:
## 🚀 Training Execution

#This cell:
#- loads config  
#- sets device  
#- loads data  
#- builds model  
#- trains for N epochs  
#- saves best checkpoint  
#- saves plots + metrics  

# Main Training Logic


In [36]:
# Example usage inside notebook:
# config = load_config("configs/experiments/convnext_t/phase1_lr1e-4_bs32.yaml")

def run_training(config):
    run_name   = config["run"]["name"]
    model_name = config["run"]["model"]

    output_dir = config["output"]["base_dir"].replace("{model}", model_name)
    output_dir = os.path.join(output_dir, run_name)
    os.makedirs(output_dir, exist_ok=True)

    print(f"Training {model_name} — {run_name}")

    set_seed(config["dataset"]["seed"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] {device}")

    save_config(config, output_dir)

    train_loader, val_loader, test_loader, class_weights = get_dataloaders(config)
    model = get_model(config).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = get_optimizer(model, config)
    scheduler = get_scheduler(optimizer, config)

    epochs = config["training"]["epochs"]
    log_interval = config["output"]["log_interval"]

    best_val_f1 = 0.0
    metrics = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": [], "val_f1": []
    }

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, log_interval)
        val_loss, val_acc, val_f1, val_preds, val_labels = validate(model, val_loader, criterion, device)
        scheduler.step()

        metrics["train_loss"].append(train_loss)
        metrics["train_acc"].append(train_acc)
        metrics["val_loss"].append(val_loss)
        metrics["val_acc"].append(val_acc)
        metrics["val_f1"].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), os.path.join(output_dir, "best_model.pth"))
            print(f"[Checkpoint] New best F1={best_val_f1:.4f}")

    save_loss_curves(metrics, output_dir)
    save_confusion_matrix(val_labels, val_preds, train_loader.dataset.classes, output_dir)
    save_metrics(metrics, output_dir)

    print("Training complete.")

# Test run

In [37]:
# Load config
config = load_config("configs/experiments/smoke_test.yaml")

# Run training
run_training(config)

Training resnet50 — smoke_test
[Seed] Set to 42
[Device] cuda
[Data] Train=7069 | Val=1754 | Test=1551
[Model] Loading resnet50 (pretrained=False)
[Model] Params — total: 23,565,404 | trainable: 23,565,404

Epoch 1/2
[Batch 10] loss=4.4093
[Batch 20] loss=3.9418
[Batch 30] loss=2.8378
[Batch 40] loss=3.1146
[Batch 50] loss=3.9212
[Batch 60] loss=3.5935
[Batch 70] loss=3.3684
[Batch 80] loss=3.8962
[Batch 90] loss=5.2340
[Batch 100] loss=3.5716
[Batch 110] loss=3.1394
[Batch 120] loss=3.5369
[Batch 130] loss=3.6347
[Batch 140] loss=3.0350
[Batch 150] loss=3.0746
[Batch 160] loss=3.0971
[Batch 170] loss=4.6672
[Batch 180] loss=3.2327
[Batch 190] loss=2.6832
[Batch 200] loss=3.6772
[Batch 210] loss=3.4055
[Batch 220] loss=2.9768
[Batch 230] loss=2.8913
[Batch 240] loss=3.4315
[Batch 250] loss=3.1551
[Batch 260] loss=3.4846
[Batch 270] loss=3.3013
[Batch 280] loss=2.8777
[Batch 290] loss=3.5780
[Batch 300] loss=3.3494
[Batch 310] loss=3.6931
[Batch 320] loss=2.9885
[Batch 330] loss=2.9847
